# Treinamento YOLOv8 — Treine Seu Primeiro Modelo (versão AO VIVO, workshop)
Ambiente: 100% Google Colab, sem Google Drive e sem instalar nada no computador do laboratório.

**Autor: Rodrigo Gossi — 2026**

⚠️ Esta é a versão usada AO VIVO no Workshop de Visão Computacional Aplicada. Este repositório (`treinamentoDL_minimo`) é um recorte enxuto, feito só para esse exercício: contém as 5 imagens de entrada, o mini-dataset já rotulado e este notebook — nada mais. Um `git clone` simples já traz tudo, sem depender de conta/pasta pessoal no Drive de cada participante.

## O que já foi feito antes de você chegar aqui
*(demonstrado ao vivo pelo apresentador, não é preciso repetir)*

As 5 imagens em [`imagens_entrada/`](imagens_entrada) foram rotuladas manualmente no **Label Studio** — uma única caixa por imagem, na única classe do projeto: `placa`. O export já saiu pronto no formato YOLO (imagem + `.txt` com a caixa normalizada) e está salvo neste mesmo repositório, em [`dataset/`](dataset).

CÉLULA 1 — Instalação

In [ ]:
!pip install -U ultralytics pyyaml

CÉLULA 2 — Baixar o material do workshop

Sem Google Drive: o repositório é público, então um `git clone` simples já traz o mini-dataset rotulado. Se a pasta já existir (por exemplo, se você rodar esta célula de novo), o clone é pulado.

In [ ]:
import os

if not os.path.isdir("treinamentoDL_minimo"):
    !git clone --quiet https://github.com/rodrigogossi/treinamentoDL_minimo.git
    print("✅ Repositório clonado.")
else:
    print("✅ Repositório já estava clonado (pulando).")

CÉLULA 3 — Configurações

In [ ]:
from pathlib import Path

#⚠️ Só mude isto se o nome da pasta clonada for diferente do padrão
raw_export_path = Path("treinamentoDL_minimo/dataset")                       # ← EDITE AQUI (export do Label Studio)
dataset_path    = Path("dataset_mini/license-plate-mini")                    # onde o split vai ficar (local, no Colab)
resultados_path = Path("resultados_treino").resolve()                        # onde os pesos treinados vão ficar (local, no Colab)
config_path     = Path("config.yaml")                                        # gerado pela CÉLULA 4, abaixo

assert raw_export_path.is_dir(), f"⚠️ Não encontrei o export do Label Studio em: {raw_export_path}"
print(f"✅ Export do Label Studio encontrado: {raw_export_path}")

CÉLULA 4 — Dividir o export em treino/val e gerar o config.yaml

O YOLO espera duas pastas separadas: `train` (as imagens que o modelo efetivamente enxerga e aprende) e `val` (imagem que ele **não** vê durante o treino, usada só pra conferir se ele está generalizando em vez de decorar). Com só 5 imagens, o split é propositalmente simples e determinístico: ordena por nome e reserva a **última** para validação — sempre a mesma, toda vez que rodar.

In [ ]:
import shutil

import yaml

SUPPORTED_EXT = [".jpg", ".jpeg", ".png"]
VAL_COUNT = 1  # com só 5 imagens, 1 já é o mínimo pra existir alguma validação

def localizar_imagens_e_labels(input_dir):
    # Aceita tanto o layout com subpastas (images/, labels/) quanto o achatado —
    # isso varia entre versões do Label Studio.
    imagens_dir = input_dir / "images" if (input_dir / "images").is_dir() else input_dir
    labels_dir = input_dir / "labels" if (input_dir / "labels").is_dir() else input_dir
    imagens = sorted(f for ext in SUPPORTED_EXT for f in imagens_dir.glob(f"*{ext}"))
    labels_por_nome = {f.stem: f for f in labels_dir.glob("*.txt")}
    return imagens, labels_por_nome

def copiar_split(nome_split, imagens, labels_por_nome):
    imagens_dst = dataset_path / "images" / nome_split
    labels_dst = dataset_path / "labels" / nome_split
    imagens_dst.mkdir(parents=True, exist_ok=True)
    labels_dst.mkdir(parents=True, exist_ok=True)
    for img in imagens:
        shutil.copy2(img, imagens_dst / img.name)
        label = labels_por_nome.get(img.stem)
        if label is not None:
            shutil.copy2(label, labels_dst / label.name)
    print(f"✅ {nome_split}: {len(imagens)} imagem(ns) copiada(s) para {imagens_dst}")

if not dataset_path.exists():
    imagens, labels_por_nome = localizar_imagens_e_labels(raw_export_path)
    assert imagens, f"⚠️ Nenhuma imagem encontrada em {raw_export_path}"
    treino, val = imagens[:-VAL_COUNT], imagens[-VAL_COUNT:]
    copiar_split("train", treino, labels_por_nome)
    copiar_split("val", val, labels_por_nome)
else:
    print(f"✅ Mini-dataset já dividido em: {dataset_path} (pulando divisão)")

# Gera o config.yaml que o Ultralytics precisa, já apontando pro dataset local
cfg = {
    "path": str(dataset_path.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "placa"},
}
with open(config_path, "w") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)
print(f"✅ config.yaml gerado em: {config_path.resolve()}")

CÉLULA 5 — Validar a estrutura (train/val)

In [ ]:
def validar_split(nome, images_dir, labels_dir):
    assert images_dir.exists(), f"Pasta não encontrada: {images_dir}"
    assert labels_dir.exists(), f"Pasta não encontrada: {labels_dir}"

    images = sorted([f for ext in SUPPORTED_EXT for f in images_dir.rglob(f"*{ext}")])
    labels = sorted(labels_dir.rglob("*.txt"))

    image_stems   = {img.stem for img in images}
    label_stems   = {lbl.stem for lbl in labels}
    pares_validos = image_stems & label_stems

    sem_label  = image_stems - label_stems
    sem_imagem = label_stems - image_stems

    print(f"--- {nome} ---")
    print(f"✅ Imagens: {len(images)} | Labels: {len(labels)} | Pares válidos: {len(pares_validos)}")
    if sem_label:
        print(f"⚠️  {len(sem_label)} imagem(ns) sem label")
    if sem_imagem:
        print(f"⚠️  {len(sem_imagem)} label(s) sem imagem")
    print()

validar_split("train", dataset_path / "images" / "train", dataset_path / "labels" / "train")
validar_split("val",   dataset_path / "images" / "val",   dataset_path / "labels" / "val")

CÉLULA 6 — Treinar

`project`/`name`/`exist_ok` só organizam onde os pesos ficam salvos (localmente, dentro deste ambiente do Colab). `epochs=100` é explícito (em vez de deixar no padrão do Ultralytics) só pra deixar claro: com 5 imagens isso ainda leva menos de 2 minutos, mesmo numa máquina sem GPU — e é o que garante que o modelo saia com confiança alta o bastante pra detectar algo de verdade daqui a pouco.

In [ ]:
from ultralytics import YOLO

# "data" é o único parâmetro realmente obrigatório do .train() — ele precisa
# saber qual dataset usar. "epochs" é explícito de propósito (ver célula
# acima): o resto (batch, otimizador...) fica no padrão do Ultralytics.
model = YOLO("yolov8n.pt")
model.train(
    data     = str(config_path),
    epochs   = 100,
    project  = str(resultados_path),
    name     = "treino",
    exist_ok = True,  # reaproveita a mesma pasta "treino" a cada execução
)

pesos_path = resultados_path / "treino" / "weights" / "best.pt"
print(f"\n✅ Pesos treinados salvos em: {pesos_path}")

CÉLULA 7 — Ver uma predição rápida

Rodamos a predição em cima de todas as 5 imagens (treino + validação) — com um dataset tão pequeno, é normal que a imagem de validação isolada às vezes não dê detecção nenhuma (ela é, de propósito, a mais difícil: a única que o modelo nunca viu). Por isso escolhemos pra mostrar a que teve a detecção com maior confiança entre as 5.

In [ ]:
modelo_treinado = YOLO(str(pesos_path))  # pesos que você acabou de treinar
imagens = [f for ext in SUPPORTED_EXT for f in (dataset_path / "images").rglob(f"*{ext}")]

# conf baixo (0.05) de propósito: mesmo bem treinado, um modelo de 5 imagens
# tem confiança naturalmente mais baixa que um modelo "de verdade" (treinado
# com milhares de imagens, como o do app mobile). O padrão do Ultralytics
# (0.25) esconderia detecções legítimas por causa disso.
resultados = modelo_treinado.predict(source=[str(f) for f in imagens], conf=0.05, save=True)

print(f"\n✅ Predições salvas em: {resultados[0].save_dir}")

# Escolhe a imagem com a detecção de maior confiança para exibir aqui.
melhor = max(resultados, key=lambda r: r.boxes.conf.max().item() if len(r.boxes) else -1)

from IPython.display import Image, display
display(Image(filename=str(Path(melhor.save_dir) / Path(melhor.path).name)))

**Você acabou de treinar seu primeiro modelo de visão computacional — parabéns!** O resultado não vai ser robusto (5 imagens são poucas demais pra isso, de propósito), mas o fluxo é exatamente o mesmo usado para treinar o modelo real que você vai ver rodando no aplicativo mobile daqui a pouco — só que com um dataset de 25 mil imagens em vez de 5, e treinando por bem mais tempo.